# Stop Writing Prompts: Let Your Best Chats Write Them for You

We don't write the system prompt. We give an optimizer a short **rubric** and **60 labelled example chats**,
and DSPy's **GEPA** optimizer writes and tests the prompt, graded by an **LLM judge**.
Then **Promptfoo** checks the new prompt against the old one on tickets the optimizer never saw.

Demo: *Maple & Co.*, a fictional home-goods shop. All tickets are synthetic.
Models: Anthropic's Claude API (`claude-haiku-5-5` runs the bot, `claude-sonnet-5-5` judges and writes new prompts).
This uses a paid API. Check the pricing page before you run.

In [1]:
# Setup. In Colab this clones the repo and installs pinned packages.
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/davolu/stop-writing-prompts.git
    %cd stop-writing-prompts
    %pip -q install -r requirements.txt
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")  # from Colab Secrets
else:
    os.chdir(os.path.dirname(os.getcwd()) if os.getcwd().endswith("notebook") else os.getcwd())
sys.path.insert(0, os.getcwd())

In [2]:
# Check the key is set. This never prints the key.
!test -n "$ANTHROPIC_API_KEY" && echo set

set


## 1. The rubric: what a good reply looks like

In [3]:
from src.config import read
print(read("rubric.md"))

# What a good Maple & Co. reply looks like

Score each criterion 0, 1 or 2.

- Empathetic: acknowledges how the customer feels in one plain, specific sentence. No grovelling, no stock phrases.
- Friendly: warm and human. Uses the customer's first name when we know it. Not corporate, not jokey.
- Straight to the point: under 90 words. Gives the one policy fact that matters and the exact next step.
- Faithful: uses only facts from the ticket and the policy. Never invents a refund, a date, a tracking number, a discount or an exception. Escalates to a team lead when the customer asks for an exception.

A reply that invents anything scores 0 on Faithful, and fails overall.


## 2. Gold chats: example replies we'd actually send

In [4]:
import json
from src.data import load_gold
gold = load_gold()
print(len(gold), "gold chats,", sum(g["edited"] for g in gold), "edited by hand")
ex = next(g for g in gold if g["id"] == "T-late")
print("\nTICKET:", ex["ticket"])
print("\nGOLD REPLY:\n" + ex["gold_reply"])

60 gold chats, 7 edited by hand

TICKET: Hello, I'd like to return a throw blanket I bought. It was delivered 41 days ago and I never used it. Grace

GOLD REPLY:
Hi Grace,

I'm sorry the blanket didn't work out. It was delivered 41 days ago, which is past our 30-day refund window, so I can't offer a refund. Because it's unused, you can still return it for store credit up to 60 days after delivery. Reply here and we'll send your return details.

Warmly,
Maple & Co. Support


In [5]:
# One of our hand edits: the draft was too long and offered an exception nobody asked for.
drafts = {d["id"]: d for d in json.load(open("data/gold_drafts.json"))}
edits = json.load(open("data/edits.json"))
print("WHY:", edits["T-late"]["why"])
print("\nDRAFT (%d words):\n%s" % (len(drafts["T-late"]["gold_reply"].split()), drafts["T-late"]["gold_reply"]))

WHY: Too long (93 words), and it offers an exception nobody asked for.

DRAFT (93 words):
Hi Grace,

Thanks for getting in touch, and it's good to hear the blanket is still unused. Since it was delivered 41 days ago, it's past our 30-day refund window, so I can't offer a refund. Unused items can still be returned for store credit up to 60 days after delivery. Reply to confirm you'd like store credit and we'll send your return details. If you'd like an exception to the refund rule, I can pass it to a team lead, who will reply within 1 business day.

Warmly,
Maple & Co. Support


## 3. Train, validation and test

In [6]:
from src.data import examples
split = json.load(open("data/splits.json"))
print({k: len(v) for k, v in split.items() if k != "seed"}, "seed =", split["seed"])
print("cracked vase is in test:", "T-vase" in split["test"])
train, val, test = examples("train"), examples("val"), examples("test")

{'train': 36, 'val': 12, 'test': 12} seed = 7
cracked vase is in test: True


## 4. Models, the baseline prompt and the judge

In [7]:
import dspy
from src.config import make_lms
from src.judge import set_judge_lm, grade
from src.signatures import BASELINE, make_bot

task_lm, judge_lm = make_lms()      # haiku-5-5 runs the bot, sonnet-5-5 judges
dspy.configure(lm=task_lm)
set_judge_lm(judge_lm)

baseline_bot = make_bot(BASELINE)
print("BASELINE PROMPT:", BASELINE)

BASELINE PROMPT: You are a helpful support agent for Maple & Co. Be nice and solve the customer's problem.


In [8]:
vase = next(e for e in test if e.id == "T-vase")
reply = baseline_bot(policy=vase.policy, ticket=vase.ticket).reply
print(f"BASELINE REPLY ({len(reply.split())} words):\n{reply}")
g = grade(vase.ticket, vase.reply, reply)
print(f"\nJUDGE score={g.score:.3f}\n{g.feedback}")

BASELINE REPLY (121 words):
Hi Ada,

I'm so sorry your ceramic vase arrived cracked. That's disappointing, and I understand you'd like this sorted quickly.

To process this for order #4812, please reply with a clear photo of the cracked vase and its packaging. Once we have the photo, you can choose either a free replacement or a full refund. You don't need to send the broken vase back.

Please note that I can't guarantee a same-day refund. If you choose a refund, it will go back to your original payment method within 5 to 10 business days of being processed.

Thank you for your patience, Ada. We'll get this resolved for you as soon as we have the photo.

Warm regards,
Maple & Co. Support

JUDGE score=0.750
[E2 F2 P1 T1, 121 words] The reply is about 120 words, well over the 90-word limit. Cut "Thank you for your patience, Ada. We'll get this resolved for you as soon as we have the photo." and shorten the refund sentence. It also adds facts the policy doesn't state: "and its packaging" 

In [9]:
from src.evaluate import run
base = run(baseline_bot, test)
print(f"Baseline test score: {base['mean']:.3f}")

Baseline test score: 0.646


## 5. Let GEPA write the prompt

In [10]:
from src.optimize import optimize, export
RUN_GEPA = False   # True re-runs the optimizer (paid). False loads the program we saved from our run.

if RUN_GEPA:
    optimized_bot = optimize(train, val, reflection_lm=judge_lm)   # auto="light"
else:
    optimized_bot = make_bot()
    optimized_bot.load("outputs/optimized_program.json")
print(open("prompts/optimized.txt").read())

You are a support agent for Maple & Co., an online home-goods shop. You receive two inputs: a support policy and a customer ticket. Write ONE complete reply to the customer, grounded strictly in the policy text supplied (verify each fact against it every time, since it may change). Replies are scored on Empathy, Faithfulness, Policy/Process, Tone/Terseness and word count (each roughly 0-2).

COMPLETENESS (critical)
- Always finish the reply: greeting, one empathy sentence, answer and next step, sign-off. A reply cut off mid-word scored very badly. Keep it compact so it finishes well within the limit. Do not start long preambles.

HARD CONSTRAINTS
- LENGTH: Under 90 words total, greeting and sign-off included. Aim for 60-80. Count before finalizing. Drafts of 115-160 words were penalized.
- No padding. Never repeat a fact. No filler such as "Thanks for reaching out about order #X", "Thanks for your understanding", "please reply if you have other questions", "happy to help get this sorte

In [11]:
s = json.load(open("outputs/gepa_summary.json"))
print("candidates tried:", s["candidates"], " winner:", s["best_idx"])
print("validation scores:", s["val_aggregate_scores"])

candidates tried: 20  winner: 19
validation scores: [0.479, 0.823, 0.896, 0.833, 0.708, 0.771, 0.521, 0.906, 0.885, 0.938, 0.938, 0.948, 0.896, 0.906, 0.917, 0.927, 0.844, 0.854, 0.875, 0.969]


## 6. Score the new prompt on the held-out test set

In [12]:
opt = run(optimized_bot, test)
print(f"Baseline test score:  {base['mean']:.3f}")
print(f"Optimized test score: {opt['mean']:.3f}")

Baseline test score:  0.646
Optimized test score: 0.854


In [13]:
reply = optimized_bot(policy=vase.policy, ticket=vase.ticket).reply
print(f"OPTIMIZED REPLY ({len(reply.split())} words):\n{reply}")
print("\nJUDGE:", grade(vase.ticket, vase.reply, reply).feedback)

OPTIMIZED REPLY (95 words):
Hi Ada,

I'm sorry your vase arrived cracked, and I know you wanted a refund today.

Damaged items qualify for a full refund or a free replacement, and you don't need to send the vase back. I can't promise a same-day refund, but refunds reach the original payment method in 5 to 10 business days. To start, reply with a photo of the crack and let me know whether you'd like a refund or a replacement for order #4812. The photo needs to be sent within 30 days of delivery.

Warmly,
Maple & Co. Support

JUDGE: [E2 F2 P1 T2, 95 words] The reply is warm and accurate. It names Ada's cracked vase and her wish for a refund today, and every fact matches the policy. It runs to about 90 words in the body, or 95 with the sign-off, so it is not under 90. Cut the last sentence, "The photo needs to be sent within 30 days of delivery." The 30-day window is a minor detail here, and dropping it brings the reply under the limit with a clearer single next step.


## 7. Regression test with Promptfoo

In [14]:
from src.export_promptfoo import export as export_pf
print("tests:", export_pf())
RUN_PROMPTFOO = False   # True re-runs the eval (paid). False reads the results we saved from our run.
if RUN_PROMPTFOO:
    !cd promptfoo && npx -y promptfoo@latest eval --no-progress-bar -o ../outputs/promptfoo_results.json
    from src.promptfoo_usage import record as record_pf; record_pf()
pf = json.load(open("outputs/promptfoo_results.json"))["results"]
for p in pf["prompts"]:
    m = p["metrics"]
    print(f'{p["label"]:10} passed {m["testPassCount"]:2}/12   under_90_words {m["namedScores"]["under_90_words"]:.0f}/12   rubric avg {m["namedScores"]["rubric"]/12:.2f}')

tests: 12
baseline   passed  1/12   under_90_words 1/12   rubric avg 0.59
optimized  passed 11/12   under_90_words 12/12   rubric avg 0.91


## 8. What it cost (from the API usage fields)

In [15]:
from src import usage
usage.record("6_notebook_run", [task_lm, judge_lm])   # calls this notebook made (cached calls are free)
t = usage.total()
usage.print_usage(t)
print(f"TOTAL ${sum(r['cost_usd'] for r in t.values()):.2f}")

claude-haiku-5-5   calls= 564  in=2,045,687  out=389,497  $0.3993
claude-sonnet-5-5  calls= 542  in=1,030,601  out=293,180  $4.9929
TOTAL $5.39
